# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Note:** All entities—record sets, fields, columns—are referenced by their `@id` values.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset (metadata & structure)
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Dataset Description: {metadata.description}")
print(f"Dataset URL: {croissant_url}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s for this dataset.

We'll fetch the record sets defined in the schema, list their `@id`s, and inspect their fields and column `@id`s.

In [ ]:
# List available record sets and their fields
record_sets = dataset.metadata.recordSet

if not record_sets:
    print("No record sets found in metadata. Please check schema or record set structure.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        print("  Fields:")
        for fld in rs.get('field', []):
            print(f"    - {fld['@id']} (name: {fld.get('name', '')})")
        print("  Columns:")
        for col in rs.get('column', []):
            print(f"    - {col['@id']} (name: {col.get('name', '')})")
        print()

### Example: Previewing some records (by record set `@id`)

You may select a record set `@id` from above and print a few records with their field `@id`s.

In [ ]:
# Example: Preview first 3 records for each record set by @id
for rs in record_sets:
    rs_id = rs['@id']
    print(f"--- Records for record set {rs_id} ---")
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        print(record)
        if i >= 2:
            break
    print()

## 3. Data Extraction
Load data from *all* available record sets into pandas DataFrames, using record set and field `@id`s. We'll store DataFrames in a dictionary for easy access.

In [ ]:
dataframes = {}

record_set_ids = [rs['@id'] for rs in record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for Record Set: {rs_id}")
    print(f"Fields/Columns (by @id): {df.columns.tolist()}")
    print(df.head())
    print()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filter by criteria, normalize numeric fields, group by key attributes.

For demonstration, let's select the main survey record set and a numeric field (e.g. GAD-7 score) for analysis.

**Replace the field `@id`s below with those of the main record set and numeric fields as appropriate from your dataset's structure.**

In [ ]:
# Example Record Set and Field IDs (replace with those specific to dataset, if necessary)
main_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[main_record_set_id] if main_record_set_id else pd.DataFrame()

# Find a numeric field by inspecting DataFrame columns
# For example, suppose '@id' for numeric field is 'gad7_score', and for group field is 'village'
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'gad' in col.lower():
        numeric_field_id = col
    if 'village' in col.lower():
        group_field_id = col

if numeric_field_id:
    # Filter records where GAD-7 score > threshold (e.g., 10)
    threshold = 10
    filtered_df = df[df[numeric_field_id].astype(float) > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()

    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (e.g. village)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field matching 'gad7_score' found. Please check column names.")

## 5. Visualization
Visualize a numeric field's distribution and analyze by group (e.g., GAD-7 scores by village).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 scores (replace field id if needed)
if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].astype(float), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} scores")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot of GAD-7 scores by village
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id].astype(float))
        plt.title(f"{numeric_field_id} scores by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- Successfully loaded the dataset using the Croissant schema and `mlcroissant`.
- Inspected and extracted all record sets and fields using their `@id`.
- Explored and processed survey responses, demonstrated filtering and normalization of numeric mental health scores (e.g. GAD-7).
- Visualized distributions and grouped data by relevant demographic fields (e.g. village).

**Next steps**: Use the processed data for more advanced statistical analysis, modeling, or reporting as appropriate for research or public health stakeholders.